# M5.3 — Logistic Regression baseline B03

Milestone:

`M5 — Modeling baseline`

Substep:

`M5.3 — Logistic Regression baseline`

Official candidate runs:

`M5-LR-SHORT-B03`

`M5-LR-LONG-B03`

B03 được mở vì:

```text
B01:
SAGA + max_iter=200
→ ConvergenceWarning
→ FAILED_CONVERGENCE

B02:
SAGA + max_iter=1000
→ W_LONG chạy >25 phút nhưng chưa hoàn thành
→ COMPUTATIONALLY INEFFICIENT / ABORTED
```

B03 thay solver vì lý do:

`optimization / computational suitability`

không phải vì validation score.

M5.3 B03 vẫn giữ:

- cùng 47-column CSR representation;
- cùng C;
- cùng L2-equivalent regularization intent;
- `class_weight=None`;
- `IMBALANCE_STRATEGY=NONE`;
- cùng VALIDATION;
- cùng metric implementation;
- cùng threshold policy;
- không dùng FINAL TEST.

B03 phải được runtime-review trước khi có bất kỳ kết luận nào.

## 1. B03 exact configuration — khóa trước result

```text
Estimator:
sklearn.linear_model.LogisticRegression

solver:
newton-cholesky

C:
1.0

l1_ratio:
0.0

max_iter:
100

tol:
1e-4

fit_intercept:
True

class_weight:
None

random_state:
42
```

### Lý do kỹ thuật

Dataset có:

```text
n_samples >> n_features
```

với:

```text
W_LONG ≈ 6.86 triệu rows
feature width = 47
```

Do đó B03 dùng solver Newton-Cholesky để thử một optimization path phù hợp hơn cho bài toán ít feature nhưng rất nhiều mẫu.

`max_iter=100` được xem là iteration budget kỹ thuật cho Newton solver, không phải hyperparameter tuning theo validation score.

Nếu B03 có warning/convergence failure:

`STOP`

không được diễn giải metric.

## 2. Runtime order

B03 chạy:

```text
1. load + audit contract
2. khóa config
3. W_SHORT trước
4. convergence/warning gate
5. nếu PASS → W_LONG
6. convergence/warning gate
7. controlled-pair gate
8. persist outputs
9. final technical gate
```

W_SHORT chạy trước chỉ để giảm rủi ro lãng phí thời gian.

Không dùng W_SHORT score để sửa B03 config trước khi W_LONG.

Nếu W_SHORT fail kỹ thuật:

`STOP — không chạy W_LONG`.

In [1]:

from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
import hashlib
import json
import platform
import sys
import time
import warnings

import numpy as np

try:
    from scipy import sparse
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Thiếu scipy. Chạy `%pip install scipy scikit-learn`, "
        "Restart Kernel rồi Run All."
    ) from exc

try:
    import sklearn
    from sklearn.base import clone
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        accuracy_score,
        confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
    )
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Thiếu scikit-learn. Chạy `%pip install scipy scikit-learn`, "
        "Restart Kernel rồi Run All."
    ) from exc


print("Python:")
print(sys.version)

print("\nPlatform:")
print(platform.platform())

print("\nNumPy:")
print(np.__version__)

print("\nscikit-learn:")
print(sklearn.__version__)


Python:
3.14.6 (main, Jun 10 2026, 10:03:53) [Clang 21.0.0 (clang-2100.0.123.102)]

Platform:
macOS-26.6.2-arm64-arm-64bit-Mach-O

NumPy:
2.5.3

scikit-learn:
1.9.1


## 3. Locate canonical artifacts và M5.2 contract

In [2]:

M4_ARTIFACT_RELATIVE_DIR = (
    Path("data")
    / "processed"
    / "m4_07_baseline_ready"
)

M5_02_CONTRACT_RELATIVE_PATH = (
    Path("data")
    / "processed"
    / "m5_02_modeling_contract"
    / "m5_02_modeling_contract.json"
)

M5_03_OUTPUT_RELATIVE_DIR = (
    Path("data")
    / "processed"
    / "m5_03_logistic_regression_baseline_b03"
)


candidate_roots = [
    Path.cwd(),
    *list(Path.cwd().parents)[:5],
]

PROJECT_ROOT = None

for candidate in candidate_roots:
    candidate = candidate.resolve()

    if (
        (
            candidate
            / M4_ARTIFACT_RELATIVE_DIR
            / "manifest.json"
        ).exists()
        and
        (
            candidate
            / M5_02_CONTRACT_RELATIVE_PATH
        ).exists()
    ):
        PROJECT_ROOT = candidate
        break


if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy PROJECT_ROOT chứa "
        "M4.7 manifest và M5.2 contract."
    )


ARTIFACT_DIR = PROJECT_ROOT / M4_ARTIFACT_RELATIVE_DIR
M5_02_CONTRACT_PATH = PROJECT_ROOT / M5_02_CONTRACT_RELATIVE_PATH
OUTPUT_DIR = PROJECT_ROOT / M5_03_OUTPUT_RELATIVE_DIR

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("PROJECT_ROOT:")
print(PROJECT_ROOT)

print("\nARTIFACT_DIR:")
print(ARTIFACT_DIR)

print("\nOUTPUT_DIR:")
print(OUTPUT_DIR)


PROJECT_ROOT:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening

ARTIFACT_DIR:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening/data/processed/m4_07_baseline_ready

OUTPUT_DIR:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening/data/processed/m5_03_logistic_regression_baseline_b03


## 4. Canonical constants

In [3]:

EXPECTED_OUTPUT_WIDTH = 47

EXPECTED_W_LONG_ROWS = 6_855_270
EXPECTED_W_SHORT_ROWS = 1_721_615
EXPECTED_VALIDATION_ROWS = 712_458

EXPECTED_W_LONG_FRAUD = 9_606
EXPECTED_W_SHORT_FRAUD = 2_491
EXPECTED_VALIDATION_FRAUD = 1_052

EXPECTED_NNZ = {
    "X_train_w_long": 61_900_193,
    "X_train_w_short": 15_542_381,
    "X_validation_w_long": 6_430_339,
    "X_validation_w_short": 6_430_339,
}

M4_PIPELINE_VERSION = "M4.7-baseline-v1"
M5_RUNNER_VERSION = "M5.2-shared-runner-v1"

IMBALANCE_STRATEGY = "NONE"
DEFAULT_RANDOM_STATE = 42
BASELINE_THRESHOLD_POLICY = "DEFAULT_MODEL_DECISION_RULE"

FEATURE_VERSION = "Feature Specification v1.0"
PREPROCESSING_VERSION = "Preprocessing Specification v1.0"
MATRIX_SCHEMA_VERSION = "Baseline Matrix Schema v1.0"

LR_MODEL_FAMILY = "Logistic Regression"
LR_MODEL_ID = "SKLEARN_LOGISTIC_REGRESSION"
LR_CONFIG_ID = "LR-B03-NEWTON-CHOLESKY-L2-C1"

LR_SHORT_EXPERIMENT_ID = "M5-LR-SHORT-B03"
LR_LONG_EXPERIMENT_ID = "M5-LR-LONG-B03"


## 5. Load contract + matrices

In [4]:

with open(
    ARTIFACT_DIR / "manifest.json",
    "r",
    encoding="utf-8",
) as file:
    manifest = json.load(file)

with open(
    ARTIFACT_DIR / "feature_names.json",
    "r",
    encoding="utf-8",
) as file:
    feature_names = json.load(file)

with open(
    M5_02_CONTRACT_PATH,
    "r",
    encoding="utf-8",
) as file:
    m5_02_contract = json.load(file)


assert manifest["pipeline_version"] == M4_PIPELINE_VERSION
assert manifest["output_width"] == EXPECTED_OUTPUT_WIDTH
assert manifest["matrix_dtype"] == "float32"
assert str(manifest["matrix_format"]).upper() == "CSR"
assert manifest["target_dtype"] == "int8"
assert manifest["raw_identifiers_in_X"] is False
assert manifest["target_in_X"] is False
assert manifest["final_test_used"] is False

assert len(feature_names) == EXPECTED_OUTPUT_WIDTH
assert len(set(feature_names)) == EXPECTED_OUTPUT_WIDTH

assert m5_02_contract["runner_version"] == M5_RUNNER_VERSION
assert m5_02_contract["final_test_access_allowed"] is False
assert m5_02_contract["imbalance_strategy"] == "NONE"


load_start = time.perf_counter()

X_train_w_long = sparse.load_npz(
    ARTIFACT_DIR / "X_train_w_long.npz"
)

X_train_w_short = sparse.load_npz(
    ARTIFACT_DIR / "X_train_w_short.npz"
)

X_validation_w_long = sparse.load_npz(
    ARTIFACT_DIR / "X_validation_w_long.npz"
)

X_validation_w_short = sparse.load_npz(
    ARTIFACT_DIR / "X_validation_w_short.npz"
)

y_train_w_long = np.load(
    ARTIFACT_DIR / "y_train_w_long.npy",
    allow_pickle=False,
)

y_train_w_short = np.load(
    ARTIFACT_DIR / "y_train_w_short.npy",
    allow_pickle=False,
)

y_validation = np.load(
    ARTIFACT_DIR / "y_validation.npy",
    allow_pickle=False,
)


print(
    "Load seconds:",
    round(
        time.perf_counter() - load_start,
        2,
    ),
)

print("\nM5.3 B03 ARTIFACT LOAD: COMPLETE")


Load seconds: 0.74

M5.3 B03 ARTIFACT LOAD: COMPLETE


## 6. Input integrity gate

In [5]:

matrices = {
    "X_train_w_long": X_train_w_long,
    "X_train_w_short": X_train_w_short,
    "X_validation_w_long": X_validation_w_long,
    "X_validation_w_short": X_validation_w_short,
}

expected_shapes = {
    "X_train_w_long": (
        EXPECTED_W_LONG_ROWS,
        EXPECTED_OUTPUT_WIDTH,
    ),
    "X_train_w_short": (
        EXPECTED_W_SHORT_ROWS,
        EXPECTED_OUTPUT_WIDTH,
    ),
    "X_validation_w_long": (
        EXPECTED_VALIDATION_ROWS,
        EXPECTED_OUTPUT_WIDTH,
    ),
    "X_validation_w_short": (
        EXPECTED_VALIDATION_ROWS,
        EXPECTED_OUTPUT_WIDTH,
    ),
}

for name, matrix in matrices.items():
    assert sparse.isspmatrix_csr(matrix)
    assert matrix.dtype == np.float32
    assert matrix.shape == expected_shapes[name]
    assert matrix.nnz == EXPECTED_NNZ[name]
    assert np.isfinite(matrix.data).all()

assert y_train_w_long.dtype == np.int8
assert y_train_w_short.dtype == np.int8
assert y_validation.dtype == np.int8

assert len(y_train_w_long) == EXPECTED_W_LONG_ROWS
assert len(y_train_w_short) == EXPECTED_W_SHORT_ROWS
assert len(y_validation) == EXPECTED_VALIDATION_ROWS

assert int(y_train_w_long.sum()) == EXPECTED_W_LONG_FRAUD
assert int(y_train_w_short.sum()) == EXPECTED_W_SHORT_FRAUD
assert int(y_validation.sum()) == EXPECTED_VALIDATION_FRAUD


print("M5.3 B03 INPUT INTEGRITY GATE: PASS")


M5.3 B03 INPUT INTEGRITY GATE: PASS


## 7. Canonical metric + shared runner

In [6]:

def compute_binary_metrics(
    y_true,
    y_pred,
):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    )

    tn, fp, fn, tp = cm.ravel()

    predicted_positive_count = int(tp + fp)

    return {
        "f1_fraud": float(
            f1_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0,
            )
        ),
        "recall_fraud": float(
            recall_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0,
            )
        ),
        "precision_fraud": float(
            precision_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0,
            )
        ),
        "accuracy_reference": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "predicted_positive_count":
            predicted_positive_count,
        "predicted_positive_rate":
            float(
                predicted_positive_count
                / len(y_true)
            ),
        "validation_rows":
            int(len(y_true)),
    }


def _json_safe_value(value):
    if isinstance(
        value,
        (str, int, float, bool, type(None)),
    ):
        return value

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, (list, tuple)):
        return [
            _json_safe_value(item)
            for item
            in value
        ]

    if isinstance(value, dict):
        return {
            str(key): _json_safe_value(item)
            for key, item
            in value.items()
        }

    return repr(value)


@dataclass
class ExperimentRunOutput:
    summary: Dict[str, Any]
    y_pred: np.ndarray
    risk_score: Optional[np.ndarray]
    warning_messages: List[str]


def run_lr_baseline(
    *,
    experiment_id,
    estimator,
    X_train,
    y_train,
    X_validation,
    y_validation,
    training_window_id,
):
    model = clone(estimator)

    with warnings.catch_warnings(
        record=True
    ) as caught_warnings:
        warnings.simplefilter("always")

        fit_start = time.perf_counter()

        model.fit(
            X_train,
            y_train,
        )

        fit_seconds = (
            time.perf_counter()
            - fit_start
        )

        pred_start = time.perf_counter()

        y_pred = model.predict(
            X_validation
        ).astype(
            np.int8
        )

        proba = model.predict_proba(
            X_validation
        )

        classes = np.asarray(
            model.classes_
        )

        positive_position = np.flatnonzero(
            classes == 1
        )

        assert len(positive_position) == 1

        risk_score = proba[
            :,
            positive_position[0],
        ].astype(
            np.float32
        )

        prediction_seconds = (
            time.perf_counter()
            - pred_start
        )

    warning_messages = [
        (
            f"{item.category.__name__}: "
            f"{item.message}"
        )
        for item
        in caught_warnings
    ]

    metrics = compute_binary_metrics(
        y_validation,
        y_pred,
    )

    model_params = {
        key: _json_safe_value(value)
        for key, value
        in model.get_params(
            deep=False
        ).items()
    }

    n_iter_values = [
        int(value)
        for value
        in np.asarray(
            model.n_iter_
        ).ravel()
    ]

    summary = {
        "runner_version":
            M5_RUNNER_VERSION,

        "experiment_id":
            experiment_id,

        "run_status":
            "COMPLETED",

        "model_family":
            LR_MODEL_FAMILY,

        "model_id":
            LR_MODEL_ID,

        "model_config_id":
            LR_CONFIG_ID,

        "model_class":
            model.__class__.__name__,

        "model_module":
            model.__class__.__module__,

        "model_params":
            model_params,

        "model_diagnostics": {
            "n_iter_":
                n_iter_values,

            "coef_l2_norm":
                float(
                    np.linalg.norm(
                        model.coef_
                    )
                ),

            "intercept":
                [
                    float(value)
                    for value
                    in np.asarray(
                        model.intercept_
                    ).ravel()
                ],
        },

        "training_window_id":
            training_window_id,

        "feature_version":
            FEATURE_VERSION,

        "preprocessing_version":
            PREPROCESSING_VERSION,

        "matrix_schema_version":
            MATRIX_SCHEMA_VERSION,

        "feature_count":
            int(
                X_train.shape[1]
            ),

        "training_rows":
            int(
                X_train.shape[0]
            ),

        "training_fraud_rows":
            int(
                np.asarray(
                    y_train
                ).sum()
            ),

        "validation_rows":
            int(
                X_validation.shape[0]
            ),

        "validation_fraud_rows":
            int(
                np.asarray(
                    y_validation
                ).sum()
            ),

        "imbalance_strategy":
            IMBALANCE_STRATEGY,

        "random_state":
            DEFAULT_RANDOM_STATE,

        "threshold_policy":
            BASELINE_THRESHOLD_POLICY,

        "fit_seconds":
            float(
                fit_seconds
            ),

        "prediction_seconds":
            float(
                prediction_seconds
            ),

        "risk_score_available":
            True,

        "risk_score_kind":
            "predict_proba",

        "warning_count":
            len(
                warning_messages
            ),

        "warnings":
            warning_messages,

        "errors":
            [],

        "final_test_accessed":
            False,

        "integrity_gate_result":
            "PASS",

        "decision":
            "OPEN — REQUIRES M5.3 RUNTIME REVIEW",

        "next_action":
            "DO NOT SELECT FINAL WINNER IN M5.3",

        **metrics,
    }

    return ExperimentRunOutput(
        summary=summary,
        y_pred=y_pred,
        risk_score=risk_score,
        warning_messages=warning_messages,
    )


def assert_clean_convergence(
    output,
    *,
    expected_max_iter,
):
    summary = output.summary

    for message in summary["warnings"]:
        print(" -", message)

    assert summary["warning_count"] == 0, (
        "Có warning cần review; "
        "không được diễn giải metric."
    )

    n_iter_values = summary[
        "model_diagnostics"
    ][
        "n_iter_"
    ]

    assert n_iter_values

    assert max(n_iter_values) < expected_max_iter, (
        "n_iter_ chạm max_iter; "
        "không được coi là converged."
    )


## 8. Define + freeze B03 config

In [7]:

LR_BASELINE_CONFIG = {
    "C": 1.0,
    "solver": "newton-cholesky",
    "max_iter": 100,
    "tol": 1e-4,
    "fit_intercept": True,
    "class_weight": None,
    "random_state": DEFAULT_RANDOM_STATE,
    "l1_ratio": 0.0,
}


lr_template = LogisticRegression(
    **LR_BASELINE_CONFIG
)


actual_params = lr_template.get_params(
    deep=False
)

for key, expected in (
    LR_BASELINE_CONFIG.items()
):
    assert actual_params[key] == expected


config_lock_payload = {
    "m5_substep": "M5.3-B03",
    "model_family": LR_MODEL_FAMILY,
    "model_id": LR_MODEL_ID,
    "model_config_id": LR_CONFIG_ID,
    "config": LR_BASELINE_CONFIG,
    "reason_for_b03": (
        "B01 failed convergence; "
        "B02 saga was computationally inefficient "
        "after >25 minutes on W_LONG."
    ),
    "validation_score_used_to_choose_b03": False,
    "imbalance_strategy": IMBALANCE_STRATEGY,
    "threshold_policy": BASELINE_THRESHOLD_POLICY,
    "final_test_access": False,
}


canonical = json.dumps(
    config_lock_payload,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
)

fingerprint = hashlib.sha256(
    canonical.encode("utf-8")
).hexdigest()

config_lock_payload[
    "config_fingerprint_sha256"
] = fingerprint


CONFIG_PATH = (
    OUTPUT_DIR
    / "m5_03_lr_b03_config_lock.json"
)


with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        config_lock_payload,
        file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )


print("B03 config:")
for key in sorted(LR_BASELINE_CONFIG):
    print(
        key,
        "=",
        LR_BASELINE_CONFIG[key],
    )

print("\nConfig fingerprint:")
print(fingerprint)

print("\nM5.3 B03 PRE-RESULT CONFIG LOCK: PASS")


B03 config:
C = 1.0
class_weight = None
fit_intercept = True
l1_ratio = 0.0
max_iter = 100
random_state = 42
solver = newton-cholesky
tol = 0.0001

Config fingerprint:
bb8f8156ce04d09045e0e2e4c67012bf5c73e58b858d1f248af3c1a529b6e810

M5.3 B03 PRE-RESULT CONFIG LOCK: PASS


# 9. W_SHORT preflight official run

Đây vẫn là official B03 run.

Nếu W_SHORT có warning/convergence issue:

`STOP`

không chạy W_LONG.

In [8]:

print("=" * 72)
print("Starting:", LR_SHORT_EXPERIMENT_ID)
print("Training rows:", f"{EXPECTED_W_SHORT_ROWS:,}")
print("Training fraud:", f"{EXPECTED_W_SHORT_FRAUD:,}")


short_output = run_lr_baseline(
    experiment_id=
        LR_SHORT_EXPERIMENT_ID,
    estimator=
        lr_template,
    X_train=
        X_train_w_short,
    y_train=
        y_train_w_short,
    X_validation=
        X_validation_w_short,
    y_validation=
        y_validation,
    training_window_id=
        "W_SHORT",
)


print("\nCompleted:", LR_SHORT_EXPERIMENT_ID)
print(
    "fit_seconds:",
    round(
        short_output.summary[
            "fit_seconds"
        ],
        3,
    ),
)
print(
    "prediction_seconds:",
    round(
        short_output.summary[
            "prediction_seconds"
        ],
        3,
    ),
)
print(
    "warning_count:",
    short_output.summary[
        "warning_count"
    ],
)
print(
    "n_iter_:",
    short_output.summary[
        "model_diagnostics"
    ][
        "n_iter_"
    ],
)


assert_clean_convergence(
    short_output,
    expected_max_iter=
        LR_BASELINE_CONFIG[
            "max_iter"
        ],
)


print(
    "\nM5.3 B03 W_SHORT CONVERGENCE GATE: PASS"
)


Starting: M5-LR-SHORT-B03
Training rows: 1,721,615
Training fraud: 2,491

Completed: M5-LR-SHORT-B03
fit_seconds: 2.166
prediction_seconds: 0.02
warning_count: 1
n_iter_: [15]
 - LinAlgWarning: The inner solver of NewtonCholeskySolver stumbled upon a singular or very ill-conditioned Hessian matrix at iteration 4. It will now resort to lbfgs instead.
Further options are to use another solver or to avoid such situation in the first place. Possible remedies are removing collinear features of X or increasing the penalization strengths.
The original Linear Algebra message was:
An ill-conditioned matrix detected: slice 0 has rcond = 3.3232385732162584e-08.


AssertionError: Có warning cần review; không được diễn giải metric.

# 10. W_LONG official run

Chỉ chạy khi W_SHORT đã PASS convergence gate.

Config không được thay đổi.

In [ ]:

print("=" * 72)
print("Starting:", LR_LONG_EXPERIMENT_ID)
print("Training rows:", f"{EXPECTED_W_LONG_ROWS:,}")
print("Training fraud:", f"{EXPECTED_W_LONG_FRAUD:,}")


long_output = run_lr_baseline(
    experiment_id=
        LR_LONG_EXPERIMENT_ID,
    estimator=
        lr_template,
    X_train=
        X_train_w_long,
    y_train=
        y_train_w_long,
    X_validation=
        X_validation_w_long,
    y_validation=
        y_validation,
    training_window_id=
        "W_LONG",
)


print("\nCompleted:", LR_LONG_EXPERIMENT_ID)
print(
    "fit_seconds:",
    round(
        long_output.summary[
            "fit_seconds"
        ],
        3,
    ),
)
print(
    "prediction_seconds:",
    round(
        long_output.summary[
            "prediction_seconds"
        ],
        3,
    ),
)
print(
    "warning_count:",
    long_output.summary[
        "warning_count"
    ],
)
print(
    "n_iter_:",
    long_output.summary[
        "model_diagnostics"
    ][
        "n_iter_"
    ],
)


assert_clean_convergence(
    long_output,
    expected_max_iter=
        LR_BASELINE_CONFIG[
            "max_iter"
        ],
)


print(
    "\nM5.3 B03 W_LONG CONVERGENCE GATE: PASS"
)


## 11. Controlled-pair comparability

In [ ]:

short_summary = short_output.summary
long_summary = long_output.summary


controlled_fields = [
    "runner_version",
    "model_family",
    "model_id",
    "model_config_id",
    "model_class",
    "model_module",
    "model_params",
    "feature_version",
    "preprocessing_version",
    "matrix_schema_version",
    "feature_count",
    "validation_rows",
    "validation_fraud_rows",
    "imbalance_strategy",
    "random_state",
    "threshold_policy",
    "risk_score_kind",
    "final_test_accessed",
]


for field in controlled_fields:
    assert (
        short_summary[field]
        == long_summary[field]
    ), (
        f"Controlled field mismatch: {field}"
    )


assert short_summary[
    "training_window_id"
] == "W_SHORT"

assert long_summary[
    "training_window_id"
] == "W_LONG"


print(
    "M5.3 B03 CONTROLLED-PAIR GATE: PASS"
)


## 12. Probability/default-rule integrity

In [ ]:

for output in [
    short_output,
    long_output,
]:
    score = output.risk_score

    assert score is not None
    assert np.isfinite(score).all()
    assert np.all(score >= 0.0)
    assert np.all(score <= 1.0)

    default_pred = (
        score >= 0.5
    ).astype(
        np.int8
    )

    assert np.array_equal(
        default_pred,
        output.y_pred,
    )


print(
    "M5.3 B03 PROBABILITY / DEFAULT-RULE GATE: PASS"
)


## 13. Baseline evidence — descriptive only

In [ ]:

def print_summary(summary):
    print("Experiment:", summary["experiment_id"])
    print("Window:", summary["training_window_id"])
    print("Training rows:", f"{summary['training_rows']:,}")
    print("Training fraud:", f"{summary['training_fraud_rows']:,}")
    print("n_iter_:", summary["model_diagnostics"]["n_iter_"])
    print("fit_seconds:", summary["fit_seconds"])
    print("F1_fraud:", summary["f1_fraud"])
    print("Recall_fraud:", summary["recall_fraud"])
    print("Precision_fraud:", summary["precision_fraud"])
    print("Accuracy:", summary["accuracy_reference"])
    print(
        "TP/FP/FN/TN:",
        summary["tp"],
        summary["fp"],
        summary["fn"],
        summary["tn"],
    )
    print(
        "Predicted positive count:",
        summary["predicted_positive_count"],
    )
    print(
        "Predicted positive rate:",
        summary["predicted_positive_rate"],
    )


print("=" * 72)
print_summary(short_summary)

print("\n" + "=" * 72)
print_summary(long_summary)

print(
    "\nTraining-window winner:",
    "OPEN — M5.3 DOES NOT SELECT",
)


## 14. Persist official B03 evidence

In [ ]:

def save_output(
    output,
):
    exp_id = output.summary[
        "experiment_id"
    ]

    pred_path = (
        OUTPUT_DIR
        / f"{exp_id}__y_pred.npy"
    )

    score_path = (
        OUTPUT_DIR
        / f"{exp_id}__risk_score.npy"
    )

    summary_path = (
        OUTPUT_DIR
        / f"{exp_id}__summary.json"
    )

    np.save(
        pred_path,
        output.y_pred,
        allow_pickle=False,
    )

    np.save(
        score_path,
        output.risk_score,
        allow_pickle=False,
    )

    with open(
        summary_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            output.summary,
            file,
            ensure_ascii=False,
            indent=2,
        )

    return {
        "prediction": pred_path,
        "risk_score": score_path,
        "summary": summary_path,
    }


saved = {
    "W_SHORT": save_output(short_output),
    "W_LONG": save_output(long_output),
}


for window_id, paths in saved.items():
    print(window_id)

    for key, path in paths.items():
        print(" ", key, "→", path)
        assert path.exists()
        assert path.stat().st_size > 0


print(
    "\nM5.3 B03 PERSISTENCE GATE: PASS"
)


## 15. Round-trip + pair manifest

In [ ]:

for window_id, output in [
    ("W_SHORT", short_output),
    ("W_LONG", long_output),
]:
    paths = saved[window_id]

    pred = np.load(
        paths["prediction"],
        allow_pickle=False,
    )

    score = np.load(
        paths["risk_score"],
        allow_pickle=False,
    )

    assert np.array_equal(
        pred,
        output.y_pred,
    )

    assert np.array_equal(
        score,
        output.risk_score,
    )

    with open(
        paths["summary"],
        "r",
        encoding="utf-8",
    ) as file:
        summary = json.load(file)

    assert (
        summary["experiment_id"]
        == output.summary[
            "experiment_id"
        ]
    )


pair_manifest = {
    "m5_substep": "M5.3-B03",
    "model_family": LR_MODEL_FAMILY,
    "model_config_id": LR_CONFIG_ID,
    "config_fingerprint_sha256": fingerprint,
    "b01_status": "FAILED_CONVERGENCE",
    "b02_status": "ABORTED_COMPUTATIONALLY_INEFFICIENT",
    "b03_status": "COMPLETED_PENDING_RUNTIME_REVIEW",
    "training_window_winner": "OPEN",
    "final_threshold": "OPEN",
    "final_test_accessed": False,
    "runs": {
        "W_SHORT": LR_SHORT_EXPERIMENT_ID,
        "W_LONG": LR_LONG_EXPERIMENT_ID,
    },
}


PAIR_MANIFEST_PATH = (
    OUTPUT_DIR
    / "m5_03_lr_b03_pair_manifest.json"
)


with open(
    PAIR_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        pair_manifest,
        file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )


assert PAIR_MANIFEST_PATH.exists()

print(
    "M5.3 B03 ROUND-TRIP / PAIR MANIFEST GATE: PASS"
)


## 16. FINAL TEST isolation

In [ ]:

assert manifest["final_test_used"] is False
assert m5_02_contract["final_test_access_allowed"] is False

assert short_summary[
    "final_test_accessed"
] is False

assert long_summary[
    "final_test_accessed"
] is False

assert pair_manifest[
    "final_test_accessed"
] is False


for path in OUTPUT_DIR.iterdir():
    assert (
        "final_test"
        not in path.name.lower()
    )


print(
    "M5.3 B03 FINAL TEST ISOLATION GATE: PASS"
)


## 17. Overall technical gate

In [ ]:

gates = {
    "G01_INPUT_CONTRACT": True,
    "G02_B03_CONFIG_LOCK": True,
    "G03_W_SHORT_CONVERGED": True,
    "G04_W_LONG_CONVERGED": True,
    "G05_CONTROLLED_PAIR": True,
    "G06_PROBABILITY_DEFAULT_RULE": True,
    "G07_PERSISTENCE": True,
    "G08_ROUND_TRIP": True,
    "G09_FINAL_TEST_ISOLATION": True,
}


for name, value in gates.items():
    print(
        name,
        "→",
        "PASS" if value else "FAIL",
    )


assert all(gates.values())


print(
    "\nM5.3 B03 OVERALL TECHNICAL GATE: PASS"
)


# 18. Runtime review registry

Sau khi Run All, cần review:

## B03-F01 — W_SHORT optimization behavior

Cần đọc:

- fit_seconds;
- `n_iter_`;
- warning count.

Status:

`OPEN — CHỜ RUNTIME REVIEW`

---

## B03-F02 — W_LONG optimization behavior

Cần đọc:

- fit_seconds;
- `n_iter_`;
- warning count.

Status:

`OPEN — CHỜ RUNTIME REVIEW`

---

## B03-F03 — Controlled pair

Cần xác minh solver/config hoàn toàn giống nhau.

Status:

`OPEN — CHỜ RUNTIME REVIEW`

---

## B03-F04 — Baseline fraud behavior

Cần đọc F1/Recall/Precision/CM nhưng chưa chọn winner.

Status:

`OPEN — CHỜ RUNTIME REVIEW`

---

## B03-F05 — Computational suitability

Cần so runtime B03 với B01/B02 về mặt kỹ thuật.

Không dùng runtime một mình để chọn final model.

Status:

`OPEN — CHỜ RUNTIME REVIEW`

---

## B03-F06 — M5.4 handoff

Status:

`OPEN — CHỜ RUNTIME REVIEW`

# 19. Trạng thái trước runtime

```text
B01:
FAILED_CONVERGENCE

B02:
ABORTED / COMPUTATIONALLY INEFFICIENT

B03:
READY TO RUN

Solver:
newton-cholesky

C:
1.0

L2-equivalent:
l1_ratio = 0.0

max_iter:
100

Imbalance:
NONE

Threshold policy:
DEFAULT MODEL DECISION RULE

Final threshold:
OPEN

FINAL TEST:
PROTECTED

Order:
W_SHORT first
then W_LONG only if W_SHORT PASS

Training-window winner:
OPEN
```

Sau khi chạy xong, gửi lại notebook để runtime review.